Notebook para generar embeddings
Previamente ya se realizaron para pocos archivos usando un modelos de OpenAI y validando con Qdrant [rag_qdrant](https://github.com/Halsey26/embedding_PerAI/blob/main/rag_qdrant.ipynb)
Sin embargo, ahora son más de 30 archivos pdf, algunos incluso con 300 páginas. Por ende se plantea usar langchain para:
- Chunkenizado
- Embedding
- Almacenamiento - Qdrant
- Función búsqueda
Después se modularizará para detectar los pdfs y obtener los embeddings

Librerias para descargar
- %pip install -qU pypdf
- pip install langchain
- pip install langchain-community
- pip install sentence-transformers


## fsdf
Detecta si un pdf ya ha sido procesado. Si en caso no ha sido procesado, se aplica las funciones y se marca como **hecho**.

In [ ]:
import os
import hashlib

ruta_docs_pdf= '../doc_pdf'
ruta_docs_procesados= '../docs_procesados'
# carpeta_embeddings = ''

def hash_file(ruta):
    #abrimos el archivo y lo codificamos
    with open(ruta, 'rb') as file:
        return hashlib.md5(file.read())


ruta= os.path.join(ruta_docs, filename)

doc_id1= hash_file(ruta)

doc_id2 = hashlib.md5(ruta.encode()).hexdigest() # codificamos la entrada string, aplicamos algoritmo y obtenemos salida hexadecimal


In [81]:
# se crea un archivo .txt para almacenar los nombres de los archivos ya procesados
import os

if not os.path.exists('procesados.txt'):
    with open('procesados.txt', 'w') as file:
        pass # crea un archivo vacio

In [85]:
# lee los archivos no procesados, por defecto nada
with open('procesados.txt', 'r') as file:
    procesados= set(file.read().splitlines())

procesados

set()

In [89]:
docs_no_procesados= []

# recoremos los archivos pdf
for filename in os.listdir(ruta_docs_pdf):
    if filename.endswith('.pdf'):
        ruta_completa= os.path.join(ruta_docs_pdf, filename)
        print(ruta_completa)
        if filename not in procesados:
            docs_no_procesados.append(ruta_completa)

            # prueba
            with open('procesados.txt', 'w') as file:
                file.write(filename+"\n")
        

../doc_pdf/223221647-ECN-BusinessPath-fulldoc.pdf


In [90]:
docs_no_procesados

['../doc_pdf/223221647-ECN-BusinessPath-fulldoc.pdf']

In [ ]:


def hash_file(path):
    with open(path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()

def detect_new_pdfs():
    new_files = []
    for filename in os.listdir(PDF_FOLDER):
        if filename.endswith(".pdf"):
            file_path = os.path.join(PDF_FOLDER, filename)
            hash_val = hash_file(file_path)
            marker_path = os.path.join(PROCESSED_FOLDER, f"{filename}_{hash_val}.done")
            if not os.path.exists(marker_path):
                new_files.append((filename, file_path, marker_path))
    return new_files

In [2]:
import os

# Función para generar código hash para cada documento 
# def genera_hash(ruta):

# necesito verificar la lista de pdfs
for filename in os.listdir(ruta_docs):
    print(filename)
    if filename.endswith('.pdf'):
        filepath= os.path.join(ruta_docs, filename)
        print(filepath)

223221647-ECN-BusinessPath-fulldoc.pdf
../doc_pdf/223221647-ECN-BusinessPath-fulldoc.pdf


In [74]:
# generamos una función, para los id de cada documento pdf
import hashlib

ruta= os.path.join(ruta_docs, filename)

doc_id = hashlib.md5(ruta.encode()).hexdigest() # codificamos la entrada string, aplicamos algoritmo y obtenemos salida hexadecimal
doc_id



'126cf3d9d317cfba54f592249e254a09'

## Empieza el procesamiento

In [6]:
from langchain_community.document_loaders import PyPDFLoader


loader = PyPDFLoader(filepath)
pages = []
async for page in loader.alazy_load():
    pages.append(page)

In [7]:
print(f"{pages[0].metadata}\n")
print(pages[0].page_content)

{'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2013-01-30T08:10:27-08:00', 'moddate': '2013-02-05T07:38:37-05:00', 'trapped': '/False', 'source': '../doc_pdf/223221647-ECN-BusinessPath-fulldoc.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}

© EMyth 2013
The EMyth Business Development Path
1
The EMyth Business 
Development Path


In [24]:
pages[0].page_content

'© EMyth 2013\nThe EMyth Business Development Path\n1\nThe EMyth Business \nDevelopment Path'

In [23]:
import re

def clean_text(text: str) -> str:
    text = re.sub(r'©.*?\n', '', text)  # remueve símbolos de copyright y similares
    text = re.sub(r'\n+', ' ', text)  # convierte múltiples saltos de línea en espacio
    text = re.sub(r'\s{2,}', ' ', text)  # remueve espacios extra
    return text.strip()

prueba_page= pages[0].page_content
print(clean_text(prueba_page))


The EMyth Business Development Path 1 The EMyth Business Development Path


Metadata por pdf:
- documento_id
- nombre documento
- numero pagina
- total_pages

In [16]:
pages[0].metadata['total_pages']

5

In [15]:
from pathlib import Path

Path(filepath).stem

'223221647-ECN-BusinessPath-fulldoc'

In [33]:
from pathlib import Path

filename = Path(filepath).name
document_id = hashlib.md5(filename.encode()).hexdigest()
total_pages = pages[0].metadata['total_pages']
docs_metadata = []
for page in pages:
    page_number = page.metadata['page'] +1 
    
    metadata = {
        "document_id": document_id,
        "filename": filename,
        "page_number": page_number,
        "total_pages": total_pages,
    }
    cleaned_text = clean_text(page.page_content)
    
    docs_metadata.append(
        {
            'text': cleaned_text, 
            'metadata': metadata
        }
    )


In [34]:
docs_metadata

[{'text': 'The EMyth Business Development Path 1 The EMyth Business Development Path',
  'metadata': {'document_id': '7902175b59d0920eee5945c624f09c9f',
   'filename': '223221647-ECN-BusinessPath-fulldoc.pdf',
   'page_number': 1,
   'total_pages': 5}},
 {'text': 'The EMyth Business Development Path 2 Y ou’re on the road to transforming your business and yourself as a business leader. Here’s what you can expect along the way. EMyth Coaching is a relationship. It’s a place where you go for guidance, support, and challenge so you can figure out what’s in the way of creating the business you want. But its not just about having a trusted guide, they also have to have the right map. This way, you will have an orchestrated way to get the result you came for. In this case, the result is a new footing for yourself as a business leader and a fully developed business that works holistically through the 7 Dynamics. We know that you need strategy, structure and sys- tems — a different way of doing

In [36]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv()
api_key=os.getenv('OPENAI_API_KEY')
# api_key
cliente= OpenAI()
cliente

In [62]:
from sentence_transformers import SentenceTransformer

docs_embedd = []
modelo_seleccionado= SentenceTransformer('BAAI/bge-large-en-v1.5')

modelo_openai = "text-embedding-3-small"

for doc in docs_metadata:
    '''
     cuando tenga el modelo habilitado:
    response = cliente.embeddings.create(
        input= doc['text'], 
        model = modelo_openai
    )

    embedding= response.data[0].embedding
    '''
    embedding= modelo_seleccionado.encode(doc['text'], normalize_embeddings= True)
    docs_embedd.append({
        'vector': embedding.tolist(),  #con openai, directamente el embedding
        'text': doc['text'], 
        'metadata': doc['metadata']
    })

/home/codespace/.local/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


Ya ahora que tengo el embedding demo vamos a modularizar

In [66]:
docs_embedd

[{'vector': [0.0497138611972332,
   0.04694351926445961,
   -0.044723354279994965,
   -0.022731542587280273,
   -0.06637521833181381,
   -0.02240157686173916,
   0.017499852925539017,
   -0.015371651388704777,
   0.02037905342876911,
   0.04144832491874695,
   -0.01189467590302229,
   0.012864307500422001,
   0.04108339920639992,
   -0.002206462202593684,
   -0.01731467805802822,
   0.01215458009392023,
   -0.052837684750556946,
   -0.02681773528456688,
   -0.03021322563290596,
   0.012331689707934856,
   0.011663584969937801,
   -0.021653378382325172,
   -0.06614203006029129,
   -0.037349723279476166,
   0.002462279750034213,
   0.05117793753743172,
   0.00537277664989233,
   0.021219402551651,
   0.0701344758272171,
   0.02696160040795803,
   -0.019501227885484695,
   0.017046667635440826,
   0.057447779923677444,
   -0.00012350759061519057,
   -0.03930116444826126,
   0.011558870784938335,
   -0.008937196806073189,
   -0.04051511734724045,
   -0.006343390792608261,
   -0.01342871598

In [67]:
embedding[1000]

np.float32(0.0060975687)

In [43]:
from sentence_transformers import SentenceTransformer
import time

modelo_seleccionado= SentenceTransformer('BAAI/bge-large-en-v1.5')
text = 'The EMyth Business Development Path 1 The EMyth Business Development Path'
start= time.time()
embedding= modelo_seleccionado.encode(text, normalize_embeddings= True)
finish= time.time()

embedding

# text = "passage: Your text to embed"
# embedding = model.encode(text, normalize_embeddings=True)


/home/codespace/.local/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


array([ 0.04971386,  0.04694352, -0.04472335, ..., -0.02569201,
       -0.00941801,  0.01352354], shape=(1024,), dtype=float32)

In [68]:
print(f"Tiempo: {finish - start:.2f} segundos")
len(embedding)

Tiempo: 7.12 segundos


1024

In [3]:
import langchain
import langchain_community